# PyTorch Object Segmentation + ESD Line Extraction Demo

This notebook demonstrates the integration of **PyTorch-based object segmentation** with the
**LineExtraction ESD (Edge Segment Detector)** framework.

## Pipeline Overview

```
┌─────────────────────┐    ┌──────────────────────┐    ┌─────────────────────┐    ┌────────────────┐
│  Input Image        │───>│ PyTorch Segmentation │───>│  Extract Contours   │───>│  ESD + Lines   │
│  (BSDS500/MDB)      │    │  (SAM / YOLO)        │    │  (cv2.findContours) │    │  (le_edge/lsd) │
└─────────────────────┘    └──────────────────────┘    └─────────────────────┘    └────────────────┘
```

**Interactive Features:**
- **SAM (Segment Anything):** Click on any object to segment it
- **YOLO:** Automatic instance segmentation of 80 COCO classes
- **Model switching:** Toggle between SAM and YOLO in real-time
- **ESD integration:** Convert object contours to line segments

**Prerequisites:**
```bash
# Build Python bindings
bazel build //libs/...

# Install PyTorch dependencies
uv pip install torch torchvision segment-anything ultralytics
```

## 1. Setup and Imports

Configure the Python environment to use LineExtraction bindings from Bazel output directories.
Import PyTorch, segmentation models, and LE modules.

In [ ]:
import sys
import pathlib
import warnings
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional, Tuple, List, Union

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Enable interactive matplotlib backend for click events
%matplotlib widget

# --- Locate workspace root and add Bazel output dirs to sys.path ---
workspace = pathlib.Path.cwd()
while not (workspace / "MODULE.bazel").exists():
    if workspace == workspace.parent:
        raise RuntimeError("Cannot find LineExtraction workspace root (MODULE.bazel)")
    workspace = workspace.parent

# Add each binding's Bazel output directory
for lib in ["imgproc", "edge", "geometry", "eval", "lsd", "algorithm"]:
    p = workspace / f"bazel-bin/libs/{lib}/python"
    if p.exists():
        sys.path.insert(0, str(p))
    else:
        print(f"⚠ Not found: {p}  — run: bazel build //libs/{lib}/...")

# Add lsfm package for TestImages
sys.path.insert(0, str(workspace / "python"))

print(f"Workspace: {workspace}")

# Import LineExtraction modules
import le_imgproc
import le_edge
import le_geometry
import le_lsd
import le_algorithm

# Import test images helper
from lsfm.data import TestImages

# Check for PyTorch
try:
    import torch
    import torchvision
    print(f"PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    raise ImportError("PyTorch not installed. Run: uv pip install torch torchvision")

# Check for segmentation models
SAM_AVAILABLE = False
YOLO_AVAILABLE = False

try:
    from segment_anything import sam_model_registry, SamPredictor
    SAM_AVAILABLE = True
    print("✓ Segment Anything Model (SAM) available")
except ImportError:
    print("⚠ SAM not installed. Run: uv pip install segment-anything")

try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
    print("✓ Ultralytics YOLO available")
except ImportError:
    print("⚠ YOLO not installed. Run: uv pip install ultralytics")

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display, clear_output

print("\nAll modules loaded successfully.")

## 2. Model Abstraction Layer

Define a common interface for segmentation models, enabling seamless switching between SAM and YOLO.
Each model must implement:
- `load_model()` — Download and initialize the model
- `predict_mask(image, point)` — Generate binary mask from input
- `get_contours(mask)` — Extract polygon contours from mask

In [ ]:
@dataclass
class SegmentationResult:
    """Result from a segmentation model."""
    masks: List[np.ndarray]  # List of binary masks (H, W), one per detected object
    contours: List[np.ndarray]  # List of contour arrays, each shape (N, 2) as (x, y)
    labels: List[str]  # Class labels for each mask
    scores: List[float]  # Confidence scores


class SegmentationModel(ABC):
    """Abstract base class for segmentation models."""
    
    def __init__(self, device: str = "cpu"):
        self.device = device
        self.model = None
        self._loaded = False
    
    @abstractmethod
    def load_model(self) -> None:
        """Load the model weights. Called lazily on first prediction."""
        pass
    
    @abstractmethod
    def predict(
        self, 
        image: np.ndarray, 
        point: Optional[Tuple[int, int]] = None
    ) -> SegmentationResult:
        """
        Predict segmentation masks for the image.
        
        Args:
            image: RGB image as numpy array (H, W, 3), uint8
            point: Optional (x, y) click coordinate for interactive segmentation (SAM)
        
        Returns:
            SegmentationResult with masks, contours, labels, and scores
        """
        pass
    
    def ensure_loaded(self) -> None:
        """Ensure model is loaded before prediction."""
        if not self._loaded:
            print(f"Loading {self.__class__.__name__}...")
            self.load_model()
            self._loaded = True
            print(f"✓ {self.__class__.__name__} ready")
    
    @staticmethod
    def extract_contours(mask: np.ndarray, min_area: int = 100) -> List[np.ndarray]:
        """
        Extract contours from a binary mask using marching squares.
        
        Args:
            mask: Binary mask (H, W), values 0 or 255
            min_area: Minimum contour area to keep
        
        Returns:
            List of contour arrays, each shape (N, 2) as (x, y) coordinates
        """
        # Use skimage for contour extraction (more reliable than cv2 for this use case)
        from skimage import measure
        
        # Ensure binary mask
        binary = (mask > 127).astype(np.uint8)
        
        # Find contours
        contours = measure.find_contours(binary, level=0.5)
        
        result = []
        for contour in contours:
            # skimage returns (row, col) = (y, x), convert to (x, y)
            contour_xy = contour[:, ::-1].astype(np.float32)
            
            # Filter by area
            if len(contour_xy) >= 3:
                # Approximate area using shoelace formula
                x = contour_xy[:, 0]
                y = contour_xy[:, 1]
                area = 0.5 * abs(np.sum(x[:-1] * y[1:] - x[1:] * y[:-1]))
                if area >= min_area:
                    result.append(contour_xy)
        
        return result


print("SegmentationModel base class defined.")

## 3. SAM (Segment Anything Model) Wrapper

Meta's SAM enables point-based interactive segmentation. Click anywhere on an object
and SAM generates a precise mask for that object.

**Model variants:**
- `vit_h` — Huge (2.4GB) — Highest quality
- `vit_l` — Large (1.2GB) — Good balance
- `vit_b` — Base (375MB) — Fastest

In [ ]:
class SamSegmenter(SegmentationModel):
    """
    Segment Anything Model (SAM) wrapper for interactive point-based segmentation.
    
    Usage:
        sam = SamSegmenter(model_type="vit_b")
        result = sam.predict(image, point=(x, y))
    """
    
    # Model checkpoint URLs
    CHECKPOINTS = {
        "vit_h": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth",
        "vit_l": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth",
        "vit_b": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
    }
    
    def __init__(self, model_type: str = "vit_b", device: str = DEVICE):
        super().__init__(device)
        self.model_type = model_type
        self.predictor = None
        self._current_image = None
    
    def load_model(self) -> None:
        """Download and load SAM checkpoint."""
        if not SAM_AVAILABLE:
            raise ImportError("segment-anything not installed")
        
        import urllib.request
        
        # Download checkpoint if not cached
        cache_dir = workspace / ".cache" / "sam"
        cache_dir.mkdir(parents=True, exist_ok=True)
        
        checkpoint_url = self.CHECKPOINTS[self.model_type]
        checkpoint_name = checkpoint_url.split("/")[-1]
        checkpoint_path = cache_dir / checkpoint_name
        
        if not checkpoint_path.exists():
            print(f"Downloading SAM {self.model_type} checkpoint (~375MB for vit_b)...")
            urllib.request.urlretrieve(checkpoint_url, checkpoint_path)
            print(f"✓ Saved to {checkpoint_path}")
        
        # Load model
        sam = sam_model_registry[self.model_type](checkpoint=str(checkpoint_path))
        sam.to(device=self.device)
        self.predictor = SamPredictor(sam)
    
    def predict(
        self, 
        image: np.ndarray, 
        point: Optional[Tuple[int, int]] = None
    ) -> SegmentationResult:
        """
        Predict segmentation mask for clicked point.
        
        Args:
            image: RGB image (H, W, 3), uint8
            point: (x, y) click coordinate — REQUIRED for SAM
        
        Returns:
            SegmentationResult with single mask for clicked object
        """
        self.ensure_loaded()
        
        if point is None:
            # Without a point, return empty result
            return SegmentationResult(masks=[], contours=[], labels=[], scores=[])
        
        # Set image (cached for efficiency)
        if self._current_image is not image:
            self.predictor.set_image(image)
            self._current_image = image
        
        # Predict with point prompt
        input_point = np.array([[point[0], point[1]]])
        input_label = np.array([1])  # 1 = foreground
        
        masks, scores, _ = self.predictor.predict(
            point_coords=input_point,
            point_labels=input_label,
            multimask_output=True,
        )
        
        # Use highest scoring mask
        best_idx = np.argmax(scores)
        best_mask = (masks[best_idx] * 255).astype(np.uint8)
        best_score = float(scores[best_idx])
        
        # Extract contours
        contours = self.extract_contours(best_mask)
        
        return SegmentationResult(
            masks=[best_mask],
            contours=contours,
            labels=["object"],
            scores=[best_score],
        )


if SAM_AVAILABLE:
    print("SamSegmenter class defined.")
else:
    print("⚠ SamSegmenter unavailable (segment-anything not installed)")

## 4. YOLO v8 Segmentation Wrapper

Ultralytics YOLOv8-seg provides fast automatic instance segmentation for 80 COCO classes.
No user interaction required — automatically detects and segments all objects.

**Model variants:**
- `yolov8n-seg` — Nano (6.7MB) — Fastest
- `yolov8s-seg` — Small (22.4MB) — Good balance
- `yolov8m-seg` — Medium (50.5MB) — Higher accuracy

In [ ]:
class YoloSegmenter(SegmentationModel):
    """
    YOLOv8 instance segmentation wrapper for automatic object detection.
    
    Usage:
        yolo = YoloSegmenter(model_name="yolov8n-seg")
        result = yolo.predict(image)  # Returns all detected objects
    """
    
    def __init__(self, model_name: str = "yolov8n-seg", device: str = DEVICE):
        super().__init__(device)
        self.model_name = model_name
        self.conf_threshold = 0.25
    
    def load_model(self) -> None:
        """Load YOLOv8 segmentation model (auto-downloads from Ultralytics)."""
        if not YOLO_AVAILABLE:
            raise ImportError("ultralytics not installed")
        
        # YOLO auto-downloads to ~/.cache/ultralytics/
        self.model = YOLO(self.model_name)
        self.model.to(self.device)
    
    def predict(
        self, 
        image: np.ndarray, 
        point: Optional[Tuple[int, int]] = None
    ) -> SegmentationResult:
        """
        Predict segmentation masks for all detected objects.
        
        Args:
            image: RGB image (H, W, 3), uint8
            point: Optional (x, y) — if provided, only return object containing that point
        
        Returns:
            SegmentationResult with masks for all detected objects (or filtered by point)
        """
        self.ensure_loaded()
        
        # Run inference
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            results = self.model(image, conf=self.conf_threshold, verbose=False)
        
        masks = []
        contours = []
        labels = []
        scores = []
        
        if results and len(results) > 0:
            result = results[0]
            
            if result.masks is not None:
                h, w = image.shape[:2]
                
                for i, mask_data in enumerate(result.masks.data):
                    # Get mask as numpy array
                    mask = mask_data.cpu().numpy()
                    
                    # Resize to original image size if needed
                    if mask.shape != (h, w):
                        from PIL import Image as PILImage
                        mask_pil = PILImage.fromarray((mask * 255).astype(np.uint8))
                        mask_pil = mask_pil.resize((w, h), PILImage.NEAREST)
                        mask = np.array(mask_pil)
                    else:
                        mask = (mask * 255).astype(np.uint8)
                    
                    # Get class info
                    cls_id = int(result.boxes.cls[i])
                    conf = float(result.boxes.conf[i])
                    label = result.names[cls_id]
                    
                    # If point specified, check if this mask contains the point
                    if point is not None:
                        px, py = point
                        if 0 <= py < h and 0 <= px < w:
                            if mask[py, px] < 128:
                                continue  # Skip if point not in this mask
                    
                    # Extract contours
                    mask_contours = self.extract_contours(mask)
                    
                    masks.append(mask)
                    contours.extend(mask_contours)
                    labels.append(label)
                    scores.append(conf)
        
        return SegmentationResult(
            masks=masks,
            contours=contours,
            labels=labels,
            scores=scores,
        )


if YOLO_AVAILABLE:
    print("YoloSegmenter class defined.")
else:
    print("⚠ YoloSegmenter unavailable (ultralytics not installed)")

## 5. Contour-to-Lines Pipeline (Split + Fit)

This section converts segmentation contours to **real line segments** using the LineExtraction
library's `fit_line_segments()` function via the Python bindings (`le_lsd`).

**Pipeline per contour:**
1. **Pass contour points directly** to `le_lsd.fit_line_segments()`
2. **RamerSplit** — Ramer-Douglas-Peucker recursive splitting at curvature points
3. **EigenFit** — least-squares line fitting via covariance matrix eigendecomposition

This is the same split-and-fit pipeline used internally by the LSD detectors
(e.g. `LsdEL`, `LsdCC`), but operating directly on contour point sequences
rather than requiring a full edge detection image.

In [ ]:
@dataclass
class LineExtractionResult:
    """Result from line extraction pipeline."""
    contours: List[np.ndarray]  # Original contours (x, y) points
    line_segments: List[Tuple[Tuple[float, float], Tuple[float, float]]]  # All line endpoints
    lines_per_object: List[List[Tuple[Tuple[float, float], Tuple[float, float]]]]  # Grouped by contour
    raw_segments_per_object: List[list] = field(default_factory=list)  # Raw LineSegment objects per contour


def _segments_to_tuples(
    segments: list,
    min_line_length: float,
    h: int,
    w: int,
) -> List[Tuple[Tuple[float, float], Tuple[float, float]]]:
    """Convert LineSegment objects to clamped endpoint tuples."""
    result: List[Tuple[Tuple[float, float], Tuple[float, float]]] = []
    for seg in segments:
        if seg.length < min_line_length:
            continue
        sp = seg.start_point()
        ep = seg.end_point()
        x1 = max(0.0, min(float(w - 1), sp[0]))
        y1 = max(0.0, min(float(h - 1), sp[1]))
        x2 = max(0.0, min(float(w - 1), ep[0]))
        y2 = max(0.0, min(float(h - 1), ep[1]))
        result.append(((x1, y1), (x2, y2)))
    return result


class ContourToLineSegments:
    """
    Pipeline to convert segmentation contours to line segments using the
    library's RamerSplit + EigenFit algorithms.

    For each contour, we call ``le_lsd.fit_line_segments()`` which:
    1. Splits the contour at high-curvature points (Ramer-Douglas-Peucker).
    2. Fits a least-squares line to each sub-segment (EigenFit).
    3. Returns proper ``LineSegment`` objects with sub-pixel geometry.

    An optional second step refines segments with BFGS precision optimization
    via ``le_algorithm.PrecisionOptimize``.
    """

    def __init__(
        self,
        min_line_length: float = 3.0,
        split_distance: float = 2.0,
        min_pixels: int = 6,
        search_range_d: float = 1.0,
        search_range_r: float = 1.0,
    ):
        self.min_line_length = min_line_length
        self.split_distance = split_distance
        self.min_pixels = min_pixels
        # Gradient source
        self.gradient = le_imgproc.SobelGradient()
        # BFGS optimizer
        self.optimizer = le_algorithm.PrecisionOptimize({
            "search_range_d": search_range_d,
            "search_range_r": search_range_r,
        })

    def extract_lines_from_contours(
        self,
        contours: List[np.ndarray],
        image_shape: Tuple[int, int],
    ) -> LineExtractionResult:
        """
        Extract line segments from contours using RamerSplit + EigenFit.

        Args:
            contours: List of contour arrays (N, 2) as (x, y) from segmentation.
            image_shape: Original image shape (H, W) -- used for safety clamping.

        Returns:
            LineExtractionResult with per-object and aggregated line segments.
        """
        if not contours:
            return LineExtractionResult(
                contours=[], line_segments=[], lines_per_object=[],
                raw_segments_per_object=[],
            )

        h, w = image_shape
        lines_per_object: List[List[Tuple[Tuple[float, float], Tuple[float, float]]]] = []
        raw_segments_per_object: List[list] = []

        for contour in contours:
            if len(contour) < self.min_pixels:
                lines_per_object.append([])
                raw_segments_per_object.append([])
                continue

            # Call the C++ split+fit pipeline directly on contour points
            pts = contour.astype(np.int32)
            segments = le_lsd.fit_line_segments(
                pts,
                split_distance=self.split_distance,
                min_pixels=self.min_pixels,
                closed=True,
            )

            raw_segments_per_object.append(list(segments))
            obj_lines = _segments_to_tuples(segments, self.min_line_length, h, w)
            lines_per_object.append(obj_lines)

        all_lines = [seg for obj_lines in lines_per_object for seg in obj_lines]

        return LineExtractionResult(
            contours=contours,
            line_segments=all_lines,
            lines_per_object=lines_per_object,
            raw_segments_per_object=raw_segments_per_object,
        )

    def optimize_lines(
        self,
        image: np.ndarray,
        line_result: LineExtractionResult,
    ) -> LineExtractionResult:
        """
        Refine line segments with BFGS precision optimization.

        Computes gradient magnitude from the image once, then optimizes
        all segments per object to maximize gradient response.

        Args:
            image: Source image (RGB or grayscale uint8).
            line_result: Result from extract_lines_from_contours.

        Returns:
            New LineExtractionResult with optimized segments.
        """
        if not line_result.raw_segments_per_object:
            return line_result

        import cv2
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if image.ndim == 3 else image
        self.gradient.process(gray)
        magnitude = self.gradient.magnitude()

        h, w = image.shape[:2]
        opt_lines_per_object: List[List[Tuple[Tuple[float, float], Tuple[float, float]]]] = []
        opt_raw_per_object: List[list] = []

        for raw_segs in line_result.raw_segments_per_object:
            if not raw_segs:
                opt_lines_per_object.append([])
                opt_raw_per_object.append([])
                continue

            _errors, optimized = self.optimizer.optimize_all(magnitude, raw_segs)
            opt_raw_per_object.append(list(optimized))
            obj_lines = _segments_to_tuples(optimized, self.min_line_length, h, w)
            opt_lines_per_object.append(obj_lines)

        all_lines = [seg for obj_lines in opt_lines_per_object for seg in obj_lines]

        return LineExtractionResult(
            contours=line_result.contours,
            line_segments=all_lines,
            lines_per_object=opt_lines_per_object,
            raw_segments_per_object=opt_raw_per_object,
        )


# Instantiate pipeline
contour_pipeline = ContourToLineSegments()
print("ContourToLineSegments pipeline defined (split+fit + optional BFGS optimize).")

## 6. Visualization Utilities

Helper functions for displaying results at each stage of the pipeline.

In [ ]:
# Per-object color palette (distinct, colorblind-friendly)
OBJECT_COLORS_RGB = [
    (230, 25, 75),    # Red
    (60, 180, 75),    # Green
    (0, 130, 200),    # Blue
    (255, 225, 25),   # Yellow
    (245, 130, 48),   # Orange
    (145, 30, 180),   # Purple
    (70, 240, 240),   # Cyan
    (240, 50, 230),   # Magenta
    (210, 245, 60),   # Lime
    (250, 190, 212),  # Pink
]

OBJECT_COLORS_MPL = [
    tuple(c / 255 for c in rgb) for rgb in OBJECT_COLORS_RGB
]


def draw_contours_on_image(
    image: np.ndarray,
    contours: List[np.ndarray],
    color: Tuple[int, int, int] = (0, 255, 0),
    thickness: int = 2,
) -> np.ndarray:
    """Draw contours on image copy."""
    result = image.copy()
    
    for contour in contours:
        pts = contour.astype(np.int32)
        for i in range(len(pts)):
            p1 = tuple(pts[i])
            p2 = tuple(pts[(i + 1) % len(pts)])
            draw_line_on_image(result, p1, p2, color, thickness)
    
    return result


def draw_line_on_image(
    img: np.ndarray,
    p1: Tuple[int, int],
    p2: Tuple[int, int],
    color: Tuple[int, int, int],
    thickness: int = 1,
) -> None:
    """Draw a line on image using Bresenham's algorithm."""
    x0, y0 = int(p1[0]), int(p1[1])
    x1, y1 = int(p2[0]), int(p2[1])
    h, w = img.shape[:2]
    
    dx = abs(x1 - x0)
    dy = -abs(y1 - y0)
    sx = 1 if x0 < x1 else -1
    sy = 1 if y0 < y1 else -1
    err = dx + dy
    
    while True:
        for tx in range(-thickness // 2, thickness // 2 + 1):
            for ty in range(-thickness // 2, thickness // 2 + 1):
                px, py = x0 + tx, y0 + ty
                if 0 <= px < w and 0 <= py < h:
                    img[py, px] = color
        
        if x0 == x1 and y0 == y1:
            break
        
        e2 = 2 * err
        if e2 >= dy:
            err += dy
            x0 += sx
        if e2 <= dx:
            err += dx
            y0 += sy


def draw_lines_on_image(
    image: np.ndarray,
    lines: List[Tuple[Tuple[float, float], Tuple[float, float]]],
    color: Tuple[int, int, int] = (255, 0, 0),
    thickness: int = 2,
) -> np.ndarray:
    """Draw line segments on image copy."""
    result = image.copy()
    
    for p1, p2 in lines:
        draw_line_on_image(result, (int(p1[0]), int(p1[1])), (int(p2[0]), int(p2[1])), color, thickness)
    
    return result


def overlay_mask(
    image: np.ndarray,
    mask: np.ndarray,
    color: Tuple[int, int, int] = (0, 120, 255),
    alpha: float = 0.4,
) -> np.ndarray:
    """Overlay a binary mask on image with transparency."""
    result = image.copy()
    
    overlay = np.zeros_like(result)
    mask_bool = mask > 127
    overlay[mask_bool] = color
    
    result[mask_bool] = (
        (1 - alpha) * result[mask_bool] + alpha * overlay[mask_bool]
    ).astype(np.uint8)
    
    return result


def _draw_arrows_on_ax(
    ax,
    image: np.ndarray,
    lines_per_object: List[List[Tuple[Tuple[float, float], Tuple[float, float]]]],
    title: str,
):
    """Draw per-object colored arrows on an axes."""
    from matplotlib.patches import FancyArrowPatch

    ax.imshow(image)
    total = 0
    for i, obj_lines in enumerate(lines_per_object):
        mpl_color = OBJECT_COLORS_MPL[i % len(OBJECT_COLORS_MPL)]
        for (x1, y1), (x2, y2) in obj_lines:
            arrow = FancyArrowPatch(
                (x1, y1), (x2, y2),
                arrowstyle="-|>",
                mutation_scale=8,
                linewidth=1.5,
                color=mpl_color,
            )
            ax.add_patch(arrow)
            total += 1
    n_obj = len(lines_per_object)
    ax.set_title(f"{title} ({total} lines, {n_obj} objects)")
    ax.axis("off")
    return total


def visualize_pipeline_result(
    image: np.ndarray,
    seg_result: SegmentationResult,
    line_result: LineExtractionResult,
    optimized_result: Optional[LineExtractionResult] = None,
    figsize: Tuple[int, int] = (12, 10),
) -> None:
    """
    Visualize the pipeline in a 2x2 grid:

    (0,0) Segmentation masks   (0,1) Contours
    (1,0) Line Segments (raw)  (1,1) Optimized Line Segments
    """
    fig, axes = plt.subplots(2, 2, figsize=figsize)

    # --- (0,0): Segmentation masks (per-object color) ---
    if seg_result.masks:
        masked = image.copy()
        for i, mask in enumerate(seg_result.masks):
            color = OBJECT_COLORS_RGB[i % len(OBJECT_COLORS_RGB)]
            masked = overlay_mask(masked, mask, color, alpha=0.4)
        axes[0, 0].imshow(masked)
        title = "Segmentation"
        if seg_result.labels:
            title += f"\n{', '.join(seg_result.labels[:4])}"
    else:
        axes[0, 0].imshow(image)
        title = "No Segmentation"
    axes[0, 0].set_title(title)
    axes[0, 0].axis("off")

    # --- (0,1): Contours (per-object color) ---
    contour_img = image.copy()
    for i, contour in enumerate(line_result.contours):
        color = OBJECT_COLORS_RGB[i % len(OBJECT_COLORS_RGB)]
        pts = contour.astype(np.int32)
        for j in range(len(pts)):
            p1 = tuple(pts[j])
            p2 = tuple(pts[(j + 1) % len(pts)])
            draw_line_on_image(contour_img, p1, p2, color, 2)
    axes[0, 1].imshow(contour_img)
    axes[0, 1].set_title(f"Contours ({len(line_result.contours)} shapes)")
    axes[0, 1].axis("off")

    # --- (1,0): Raw line segments as arrows ---
    _draw_arrows_on_ax(axes[1, 0], image, line_result.lines_per_object, "Line Segments")

    # --- (1,1): Optimized line segments as arrows ---
    if optimized_result is not None:
        _draw_arrows_on_ax(axes[1, 1], image, optimized_result.lines_per_object, "Optimized Segments")
    else:
        axes[1, 1].imshow(image)
        axes[1, 1].set_title("Optimized Segments\n(not yet computed)")
        axes[1, 1].axis("off")

    plt.tight_layout()
    plt.show()


print("Visualization utilities defined.")

## 7. Interactive Demo Application

The interactive application provides:
- **Image selection** from BSDS500/MDB datasets
- **Model switching** between SAM and YOLO
- **Click-to-segment** for SAM (click on any object)
- **Auto-detect** for YOLO (detects all objects)
- **Real-time line extraction** on segmented objects

In [ ]:
class InteractiveSegmentationDemo:
    """
    Interactive demo combining PyTorch segmentation with ESD line extraction.
    
    Features:
    - Switch between SAM and YOLO models
    - Click on image to segment (SAM mode)
    - Automatic segmentation (YOLO mode)
    - Real-time line extraction from contours
    - Optional BFGS precision optimization step
    """
    
    def __init__(self):
        self.test_images = TestImages()
        self.current_image: Optional[np.ndarray] = None
        self.current_path: Optional[pathlib.Path] = None
        
        # Models (lazy-loaded)
        self._sam: Optional[SamSegmenter] = None
        self._yolo: Optional[YoloSegmenter] = None
        self.current_model = "yolo"  # Default to YOLO (no click required)
        
        # Results
        self.seg_result: Optional[SegmentationResult] = None
        self.line_result: Optional[LineExtractionResult] = None
        self.optimized_result: Optional[LineExtractionResult] = None
        
        # Pipeline
        self.pipeline = ContourToLineSegments()
        
        # UI elements
        self.fig = None
        self.ax = None
        self.output = widgets.Output()
        
    @property
    def sam(self) -> SamSegmenter:
        if self._sam is None:
            if not SAM_AVAILABLE:
                raise ImportError("SAM not available")
            self._sam = SamSegmenter()
        return self._sam
    
    @property
    def yolo(self) -> YoloSegmenter:
        if self._yolo is None:
            if not YOLO_AVAILABLE:
                raise ImportError("YOLO not available")
            self._yolo = YoloSegmenter()
        return self._yolo
    
    def load_image(self, path: Union[str, pathlib.Path]) -> np.ndarray:
        """Load image from path."""
        path = pathlib.Path(path)
        img = Image.open(path).convert("RGB")
        self.current_image = np.array(img)
        self.current_path = path
        
        # Reset results
        self.seg_result = None
        self.line_result = None
        self.optimized_result = None
        
        return self.current_image
    
    def get_available_images(self, dataset: str = "bsds500", limit: int = 500) -> List[pathlib.Path]:
        """Get list of available images from dataset."""
        if dataset == "bsds500":
            return list(self.test_images.bsds500())[:limit]
        elif dataset == "york_urban":
            return list(self.test_images.york_urban())[:limit]
        elif dataset == "wireframe":
            return list(self.test_images.wireframe())[:limit]
        elif dataset == "mdb":
            scenes = list(self.test_images.stereo_scenes("Q"))[:limit]
            result: List[pathlib.Path] = []
            for scene in scenes:
                for rel in (
                    f"MDB/MiddEval3-Q/{scene}/im0.png",
                    f"MDB/MiddEval3-Q/{scene}.png",
                ):
                    try:
                        result.append(self.test_images.get(rel))
                        break
                    except FileNotFoundError:
                        continue
            return result
        elif dataset == "noise":
            return list(self.test_images.noise_images())
        else:
            return []
    
    def segment(self, point: Optional[Tuple[int, int]] = None) -> SegmentationResult:
        """Run segmentation on current image."""
        if self.current_image is None:
            raise ValueError("No image loaded")
        
        if self.current_model == "sam":
            if point is None:
                return SegmentationResult(masks=[], contours=[], labels=[], scores=[])
            self.seg_result = self.sam.predict(self.current_image, point)
        else:  # yolo
            self.seg_result = self.yolo.predict(self.current_image, point)
        
        # Reset optimization when re-segmenting
        self.optimized_result = None
        
        return self.seg_result
    
    def extract_lines(self) -> LineExtractionResult:
        """Extract lines from segmentation result."""
        if self.seg_result is None or not self.seg_result.contours:
            self.line_result = LineExtractionResult(
                contours=[], line_segments=[], lines_per_object=[],
                raw_segments_per_object=[],
            )
        else:
            h, w = self.current_image.shape[:2]
            self.line_result = self.pipeline.extract_lines_from_contours(
                self.seg_result.contours, (h, w)
            )
        
        return self.line_result

    def optimize_lines(self) -> LineExtractionResult:
        """Optimize extracted line segments with BFGS."""
        if self.line_result is None or not self.line_result.raw_segments_per_object:
            return self.line_result
        
        self.optimized_result = self.pipeline.optimize_lines(
            self.current_image, self.line_result
        )
        return self.optimized_result

    def _show_results(self) -> None:
        """Display current results in the output area."""
        with self.output:
            clear_output(wait=True)
            if self.current_image is not None and self.seg_result is not None:
                visualize_pipeline_result(
                    self.current_image,
                    self.seg_result,
                    self.line_result,
                    self.optimized_result,
                )
    
    def run_pipeline(self, point: Optional[Tuple[int, int]] = None) -> None:
        """Run full pipeline and display results."""
        self.segment(point)
        self.extract_lines()
        self._show_results()
    
    def on_click(self, event):
        """Handle mouse click on image."""
        if event.inaxes != self.ax:
            return
        
        x, y = int(event.xdata), int(event.ydata)
        print(f"Click at ({x}, {y})")
        
        self.run_pipeline(point=(x, y))
    
    def create_ui(self):
        """Create the interactive UI."""
        # Dataset selector
        dataset_dropdown = widgets.Dropdown(
            options=["bsds500", "york_urban", "wireframe", "mdb", "noise"],
            value="bsds500",
            description="Dataset:",
        )
        
        # Image selector — use Select widget for scrollable list
        image_select = widgets.Select(
            options=[],
            rows=12,
            description="Image:",
            layout=widgets.Layout(width="350px"),
        )
        
        # Model selector
        model_toggle = widgets.ToggleButtons(
            options=["YOLO (Auto)", "SAM (Click)"],
            value="YOLO (Auto)",
            description="Model:",
        )
        
        # Buttons
        load_btn = widgets.Button(description="Load Image", button_style="primary")
        detect_btn = widgets.Button(description="Detect Objects", button_style="success")
        optimize_btn = widgets.Button(description="Optimize Lines", button_style="warning")
        
        # Status
        status = widgets.HTML(value="<i>Select an image to begin</i>")
        
        def update_images(change):
            paths = self.get_available_images(change["new"])
            image_select.options = [(p.name, p) for p in paths]
        
        def on_load(btn):
            path = image_select.value
            if path:
                self.load_image(path)
                status.value = f"<b>Loaded:</b> {path.name} ({self.current_image.shape[1]}x{self.current_image.shape[0]})"
                
                # Show image
                with self.output:
                    clear_output(wait=True)
                    plt.figure(figsize=(8, 6))
                    plt.imshow(self.current_image)
                    plt.title(f"{path.name} - Click to segment (SAM) or press Detect (YOLO)")
                    plt.axis("off")
                    plt.show()
        
        def on_detect(btn):
            if self.current_image is None:
                status.value = "<span style='color:red'>Load an image first!</span>"
                return
            
            status.value = "<i>Detecting...</i>"
            self.run_pipeline()
            n_obj = len(self.seg_result.masks) if self.seg_result else 0
            n_lines = len(self.line_result.line_segments) if self.line_result else 0
            status.value = (
                f"<b>Detected:</b> {n_obj} objects, {n_lines} lines "
                f"— press <em>Optimize Lines</em> to refine"
            )

        def on_optimize(btn):
            if self.line_result is None or not self.line_result.raw_segments_per_object:
                status.value = "<span style='color:red'>Run detection first!</span>"
                return

            status.value = "<i>Optimizing...</i>"
            self.optimize_lines()
            self._show_results()
            n_lines = len(self.optimized_result.line_segments) if self.optimized_result else 0
            status.value = f"<b>Optimized:</b> {n_lines} line segments refined with BFGS"
        
        def on_model_change(change):
            if "Auto" in change["new"]:
                self.current_model = "yolo"
            else:
                self.current_model = "sam"
        
        # Connect callbacks
        dataset_dropdown.observe(update_images, names="value")
        load_btn.on_click(on_load)
        detect_btn.on_click(on_detect)
        optimize_btn.on_click(on_optimize)
        model_toggle.observe(on_model_change, names="value")
        
        # Initialize image list
        update_images({"new": "bsds500"})
        
        # Layout
        controls = widgets.VBox([
            widgets.HBox([dataset_dropdown, image_select, load_btn]),
            widgets.HBox([model_toggle, detect_btn, optimize_btn]),
            status,
        ])
        
        return widgets.VBox([controls, self.output])


print("InteractiveSegmentationDemo created.")

# Create demo instance
demo = InteractiveSegmentationDemo()

### Run the Interactive Demo

Execute the cell below to launch the interactive UI. 

**How to use:**
1. Select a dataset (BSDS500 recommended for diverse images)
2. Choose an image from the dropdown
3. Click **Load Image** to display it
4. Choose model: **YOLO** for automatic detection, **SAM** for click-to-segment
5. Click **Detect Objects** to run the pipeline (2×2 grid: segmentation, contours, raw lines, placeholder)
6. Click **Optimize Lines** to refine segments with BFGS precision optimization (fills in the 4th panel)

In [ ]:
# Launch the interactive demo
ui = demo.create_ui()
display(ui)

## 8. Quick Demo (Non-Interactive)

For testing without the interactive UI, run this cell to process a single image.

In [ ]:
# Quick demo: Load a test image and run the full pipeline

# Use a Wireframe image — an architectural scene with clear line structure
test_images = TestImages()
try:
    img_path = test_images.wireframe("00037212.png")
except FileNotFoundError:
    # Fall back to first available image from any dataset
    available = list(test_images.wireframe()) or list(test_images.bsds500())
    img_path = available[0] if available else None

if img_path is not None and img_path.exists():
    print(f"Loading: {img_path.name}")

    image = np.array(Image.open(img_path).convert("RGB"))
    print(f"Image shape: {image.shape}")

    # Run YOLO segmentation (if available)
    if YOLO_AVAILABLE:
        print("\nRunning YOLO segmentation...")
        yolo = YoloSegmenter()
        seg_result = yolo.predict(image)
        print(f"Detected {len(seg_result.masks)} objects: {seg_result.labels}")

        # Extract lines from ALL detected shapes
        print("\nExtracting lines from all shapes...")
        pipeline = ContourToLineSegments()
        line_result = pipeline.extract_lines_from_contours(
            seg_result.contours,
            image.shape[:2]
        )
        for i, obj_lines in enumerate(line_result.lines_per_object):
            print(f"  Shape {i}: {len(obj_lines)} line segments")
        print(f"Total: {len(line_result.line_segments)} line segments")

        # Optimize with BFGS
        print("\nOptimizing with BFGS precision optimizer...")
        optimized_result = pipeline.optimize_lines(image, line_result)
        for i, obj_lines in enumerate(optimized_result.lines_per_object):
            print(f"  Shape {i}: {len(obj_lines)} optimized segments")
        print(f"Total: {len(optimized_result.line_segments)} optimized segments")

        # Visualize (2x2: Segmentation, Contours, Lines, Optimized)
        visualize_pipeline_result(image, seg_result, line_result, optimized_result)
    elif SAM_AVAILABLE:
        print("\nYOLO not available, using SAM with center point...")
        h, w = image.shape[:2]
        center = (w // 2, h // 2)

        sam = SamSegmenter()
        seg_result = sam.predict(image, point=center)
        print(f"Segmented object at center: score={seg_result.scores[0]:.2f}")

        # Extract lines
        pipeline = ContourToLineSegments()
        line_result = pipeline.extract_lines_from_contours(
            seg_result.contours,
            image.shape[:2]
        )
        print(f"Extracted {len(line_result.line_segments)} line segments")

        # Optimize with BFGS
        print("\nOptimizing with BFGS precision optimizer...")
        optimized_result = pipeline.optimize_lines(image, line_result)
        print(f"Optimized {len(optimized_result.line_segments)} segments")

        visualize_pipeline_result(image, seg_result, line_result, optimized_result)
    else:
        print("⚠ Neither YOLO nor SAM available. Install: uv pip install ultralytics segment-anything torch")
else:
    print("⚠ No test images found. Ensure York Urban or BSDS500 dataset is available.")
    print("  Run: ./tools/scripts/setup_york_urban.sh")

## Summary

This notebook demonstrated an end-to-end pipeline that combines **PyTorch-based object
segmentation** with the **LineExtraction** library's line fitting and precision optimization
algorithms. Starting from a raw image, objects are segmented with SAM or YOLO, their mask
contours are extracted, split at curvature points, fitted with least-squares lines, and
finally refined via BFGS gradient-based optimization for sub-pixel accuracy.

### Pipeline Components

| Stage | Component | Description |
|-------|-----------|-------------|
| **Segmentation** | SAM / YOLO | Neural network generates pixel-accurate instance masks |
| **Contour Extraction** | skimage | Marching squares extracts ordered polygon contours from binary masks |
| **Split + Fit** | `le_lsd.fit_line_segments` | Ramer-Douglas-Peucker recursive splitting followed by EigenFit least-squares line fitting |
| **Precision Optimization** | `le_algorithm.PrecisionOptimize` | BFGS optimization that maximizes gradient magnitude response to refine segment position |

### Key Classes

- `SegmentationModel` — Abstract base for segmentation models
- `SamSegmenter` — Segment Anything Model wrapper (interactive click-to-segment)
- `YoloSegmenter` — YOLOv8 instance segmentation (automatic, 80 COCO classes)
- `ContourToLineSegments` — Two-stage pipeline: `extract_lines_from_contours()` for split+fit, `optimize_lines()` for BFGS refinement

### Extensions

1. **Add more models:** Implement `SegmentationModel` for other architectures (DeepLabV3, Mask R-CNN)
2. **Tune parameters:** Adjust `split_distance`, `min_pixels`, `search_range_d/r` for different image types
3. **Batch processing:** Process multiple images and aggregate statistics
4. **Export results:** Save line segments as SVG or other vector formats

### Dependencies

```bash
# Core
bazel build //libs/...  # Build LE Python bindings

# PyTorch ecosystem
uv pip install torch torchvision segment-anything ultralytics
```